# Model 2 — Random Forest Regressor

This notebook trains **Model 2: Random Forest Regression** using the same Manhattan + Queens train/test split and feature set used for the naive baseline and Model 1.

**No hyperparameter tuning is performed yet.** This first run establishes the Random Forest result for a fair three-model comparison.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
print("Libraries imported successfully.")


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Libraries imported successfully.


## 1. Locate the repository and load the prepared data


In [3]:
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "models" / "modeling" / "train.csv").exists():
            return path
    raise FileNotFoundError(
        "Could not find models/modeling/train.csv. Run modeling_preparation_baseline.ipynb first."
    )

REPO_ROOT = find_repo_root()
MODELING_DIR = REPO_ROOT / "models" / "modeling"

TRAIN_PATH = MODELING_DIR / "train.csv"
TEST_PATH = MODELING_DIR / "test.csv"
BASELINE_METRICS_PATH = MODELING_DIR / "baseline_metrics.csv"
MODEL_1_METRICS_PATH = MODELING_DIR / "model_1_metrics.csv"

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["pickup_hour"])
test_df = pd.read_csv(TEST_PATH, parse_dates=["pickup_hour"])

print("Repository:", REPO_ROOT)
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train period:", train_df["pickup_hour"].min(), "->", train_df["pickup_hour"].max())
print("Test period :", test_df["pickup_hour"].min(), "->", test_df["pickup_hour"].max())


Repository: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting
Train shape: (460920, 17)
Test shape : (115368, 17)
Train period: 2025-09-08 00:00:00 -> 2026-01-25 03:00:00
Test period : 2026-01-25 04:00:00 -> 2026-02-28 23:00:00


## 2. Define target and features

The feature definition is intentionally the same as Model 1 so that the algorithm is the main change.

**Categorical:** `PULocationID`, `hour`, `day_of_week`, `month`

**Numerical:** weather variables, `is_holiday`, lag features, and rolling demand features.


In [4]:
TARGET = "trip_count"

CATEGORICAL_FEATURES = [
    "PULocationID",
    "hour",
    "day_of_week",
    "month",
]

NUMERICAL_FEATURES = [
    "apparent_temperature",
    "rain",
    "snowfall",
    "precipitation_probability",
    "is_holiday",
    "trip_count_lag_1h",
    "trip_count_lag_24h",
    "trip_count_lag_168h",
    "trip_count_roll_3h",
    "trip_count_roll_24h",
]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

missing_required = [
    c for c in [TARGET] + FEATURES
    if c not in train_df.columns or c not in test_df.columns
]
if missing_required:
    raise ValueError(f"Missing required modelling columns: {missing_required}")

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("Categorical features:", CATEGORICAL_FEATURES)
print("Numerical features:", NUMERICAL_FEATURES)


Categorical features: ['PULocationID', 'hour', 'day_of_week', 'month']
Numerical features: ['apparent_temperature', 'rain', 'snowfall', 'precipitation_probability', 'is_holiday', 'trip_count_lag_1h', 'trip_count_lag_24h', 'trip_count_lag_168h', 'trip_count_roll_3h', 'trip_count_roll_24h']


## 3. Validate the target and chronology


In [5]:
assert train_df["pickup_hour"].max() < test_df["pickup_hour"].min(), "Train/test chronology is not valid."
assert (y_train >= 0).all(), "Training target contains negative values."
assert (y_test >= 0).all(), "Test target contains negative values."

print("Chronological split check: passed")
print("Train target range:", y_train.min(), "to", y_train.max())
print("Test target range :", y_test.min(), "to", y_test.max())


Chronological split check: passed
Train target range: 0 to 1122
Test target range : 0 to 905


## 4. Build the preprocessing + Random Forest pipeline

Initial settings are deliberately reasonable rather than heavily tuned:
- 150 trees
- `max_depth=20`
- `min_samples_leaf=2`
- `max_features='sqrt'`
- all CPU cores


In [6]:
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot),
])

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numerical", numerical_pipeline, NUMERICAL_FEATURES),
], remainder="drop")

random_forest = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model_2 = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", random_forest),
])

print(model_2)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['PULocationID', 'hour',
                                                   'day_of_week', 'month']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['apparent_temperature',
                                                   'rain

## 5. Train Model 2


In [7]:
print("Training Random Forest...")
model_2.fit(X_train, y_train)
print("Training completed.")


Training Random Forest...
Training completed.


## 6. Generate predictions


In [8]:
train_pred = np.clip(model_2.predict(X_train), 0, None)
test_pred = np.clip(model_2.predict(X_test), 0, None)

print("Train prediction range:", train_pred.min(), "to", train_pred.max())
print("Test prediction range :", test_pred.min(), "to", test_pred.max())


Train prediction range: 0.5725211760727816 to 807.0064643080725
Test prediction range : 0.5748190841422361 to 777.1362089697778


## 7. Evaluate Model 2


In [9]:
def calculate_metrics(y_true, y_pred, dataset_name):
    return {
        "dataset": dataset_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "rows": len(y_true),
    }

model_2_metrics = pd.DataFrame([
    calculate_metrics(y_train, train_pred, "train"),
    calculate_metrics(y_test, test_pred, "test"),
])

display(model_2_metrics)


,dataset,MAE,RMSE,R2,rows
0,train,6.198628,13.697910,0.968850,460920
1,test,6.799299,15.259347,0.950739,115368


## 8. Compare Model 2 with the baseline and Model 1


In [10]:
comparison_rows = []

if BASELINE_METRICS_PATH.exists():
    baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH).iloc[0]
    comparison_rows.append({
        "model": "Naive 168-hour historical baseline",
        "MAE": baseline_metrics["MAE"],
        "RMSE": baseline_metrics["RMSE"],
        "R2": baseline_metrics["R2"],
    })

if MODEL_1_METRICS_PATH.exists():
    model_1_metrics = pd.read_csv(MODEL_1_METRICS_PATH)
    model_1_test = model_1_metrics[model_1_metrics["dataset"] == "test"].iloc[0]
    comparison_rows.append({
        "model": "Model 1 - Linear Regression",
        "MAE": model_1_test["MAE"],
        "RMSE": model_1_test["RMSE"],
        "R2": model_1_test["R2"],
    })

model_2_test = model_2_metrics[model_2_metrics["dataset"] == "test"].iloc[0]
comparison_rows.append({
    "model": "Model 2 - Random Forest",
    "MAE": model_2_test["MAE"],
    "RMSE": model_2_test["RMSE"],
    "R2": model_2_test["R2"],
})

comparison = pd.DataFrame(comparison_rows)
display(comparison)


,model,MAE,RMSE,R2
0,Naive 168-hour historical baseline,9.307390,24.056583,0.877566
1,Model 1 - Linear Regression,7.486082,15.703616,0.947828
2,Model 2 - Random Forest,6.799299,15.259347,0.950739


## 9. Random Forest feature importance

This is model interpretation, not causal evidence.


In [11]:
fitted_preprocessor = model_2.named_steps["preprocessor"]
fitted_regressor = model_2.named_steps["regressor"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_regressor.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

display(importance_df.head(30))


,feature,importance
179,numerical__trip_count_lag_1h,0.239726
181,numerical__trip_count_lag_168h,0.207353
180,numerical__trip_count_lag_24h,0.181352
182,numerical__trip_count_roll_3h,0.164812
183,numerical__trip_count_roll_24h,0.106309
124,categorical__PULocationID_237,0.012575
123,categorical__PULocationID_236,0.008615
84,categorical__PULocationID_161,0.007921
64,categorical__PULocationID_132,0.006926
85,categorical__PULocationID_162,0.003464


## 10. Save model, predictions and metrics


In [12]:
prediction_output = test_df[["PULocationID", "pickup_hour", "trip_count"]].copy()
prediction_output["prediction"] = test_pred
prediction_output["error"] = prediction_output["trip_count"] - prediction_output["prediction"]
prediction_output["absolute_error"] = prediction_output["error"].abs()

metrics_output = model_2_metrics.copy()
metrics_output["model"] = "Model 2 - Random Forest"
metrics_output = metrics_output[["model", "dataset", "MAE", "RMSE", "R2", "rows"]]

MODEL_PATH = MODELING_DIR / "model_2_random_forest.joblib"
PREDICTIONS_PATH = MODELING_DIR / "model_2_predictions.csv"
METRICS_PATH = MODELING_DIR / "model_2_metrics.csv"
IMPORTANCE_PATH = MODELING_DIR / "model_2_feature_importance.csv"

joblib.dump(model_2, MODEL_PATH)
prediction_output.to_csv(PREDICTIONS_PATH, index=False)
metrics_output.to_csv(METRICS_PATH, index=False)
importance_df.to_csv(IMPORTANCE_PATH, index=False)

print("Saved:")
print(" -", MODEL_PATH)
print(" -", PREDICTIONS_PATH)
print(" -", METRICS_PATH)
print(" -", IMPORTANCE_PATH)


Saved:
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_2_random_forest.joblib
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_2_predictions.csv
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_2_metrics.csv
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_2_feature_importance.csv


# Model 2 complete

Record the **test-set MAE, RMSE and R²** and compare them with Model 1 and the 168-hour baseline. **Do not tune the Random Forest yet**; first complete the untuned three-model comparison.
